# Reto: Análisis del Progreso Mundial de Vacunación
**Dataset:** `country_vaccinations.csv`

In [ ]:
import pandas as pd

## 1. Extraer la información del archivo

In [ ]:
df = pd.read_csv('country_vaccinations.csv')
df.head()

## 2. Mostrar la estructura y tipos de datos; convertir columnas de fecha a `datetime64`

In [ ]:
df['date'] = pd.to_datetime(df['date'])
print(df.info())
display(df.dtypes)

## 3. Determinar la cantidad de vacunas aplicadas de cada compañía
Se agrupa por la columna `vaccines` tal como la reporta cada país y se suma `daily_vaccinations`.

In [ ]:
vacunas_por_compania = (
    df.groupby('vaccines', dropna=True)['daily_vaccinations']
    .sum()
    .reset_index()
    .rename(columns={'daily_vaccinations': 'total_aplicadas'})
    .sort_values('total_aplicadas', ascending=False)
)
display(vacunas_por_compania)

## 4. Obtener la cantidad total de vacunas aplicadas en todo el mundo

In [ ]:
total_mundial = df['total_vaccinations'].sum()
print(f'Total mundial de vacunas aplicadas: {total_mundial:,.0f}')

## 5. Calcular el promedio de vacunas aplicadas por país

In [ ]:
promedio_por_pais = (
    df.groupby('country')['total_vaccinations']
    .mean()
    .reset_index()
    .rename(columns={'total_vaccinations': 'promedio_vacunas'})
    .sort_values('promedio_vacunas', ascending=False)
)
display(promedio_por_pais)

## 6. Determinar la cantidad de vacunas aplicadas el día 29/01/21 en todo el mundo

In [ ]:
fecha_objetivo = pd.Timestamp('2021-01-29')
total_29ene = df.loc[df['date'] == fecha_objetivo, 'daily_vaccinations'].sum()
print(f'Vacunas aplicadas el 29/01/2021: {total_29ene:,.0f}')

## 7. Crear `conDiferencias`: datos originales + columna `diferencias` entre `daily_vaccinations` y `daily_vaccinations_raw`

In [ ]:
conDiferencias = df.assign(
    diferencias=df['daily_vaccinations'] - df['daily_vaccinations_raw']
)
display(conDiferencias[['country', 'date', 'daily_vaccinations', 'daily_vaccinations_raw', 'diferencias']].head(10))

## 8. Obtener el periodo de tiempo exacto entre el registro más reciente y el más antiguo

In [ ]:
fecha_min = df['date'].min()
fecha_max = df['date'].max()
periodo = fecha_max - fecha_min
print(f'Fecha más antigua : {fecha_min.date()}')
print(f'Fecha más reciente: {fecha_max.date()}')
print(f'Periodo exacto    : {periodo.days} días')

## 9. Crear `conCantidad`: datos originales + columna `canVac` con la cantidad de vacunas distintas usadas cada día
Se cuenta separando los valores de la columna `vaccines` por el carácter `,`.

In [ ]:
conCantidad = df.assign(
    canVac=df['vaccines'].str.split(',').str.len()
)
display(conCantidad[['country', 'date', 'vaccines', 'canVac']].head(10))

## 10. Generar `antes20`: registros anteriores al 20 de diciembre de 2020

In [ ]:
antes20 = df.loc[df['date'] < pd.Timestamp('2020-12-20')].reset_index(drop=True)
print(f'Registros encontrados: {len(antes20)}')
display(antes20.head())

## 11. Obtener `pfizer`: registros donde se haya utilizado la vacuna Pfizer

In [ ]:
pfizer = df.loc[df['vaccines'].str.contains('Pfizer', na=False)].reset_index(drop=True)
print(f'Registros con Pfizer: {len(pfizer)}')
display(pfizer.head())

## 12. Exportar `conDiferencias`, `conCantidad`, `antes20` y `pfizer` al archivo `resultadosReto.xlsx`
Cada dataframe en una hoja distinta usando `pd.ExcelWriter`.

In [ ]:
hojas = {
    'conDiferencias': conDiferencias,
    'conCantidad'   : conCantidad,
    'antes20'       : antes20,
    'pfizer'        : pfizer,
}

with pd.ExcelWriter('resultadosReto.xlsx', engine='openpyxl') as writer:
    for nombre_hoja, dataframe in hojas.items():
        dataframe.to_excel(writer, sheet_name=nombre_hoja, index=False)

print('Archivo resultadosReto.xlsx generado exitosamente.')